In [ ]:
# Import packages, load the study configuration, the annotated sample table and the
# SigProfiler output, and set the output folder
import os
from pathlib import Path

import pandas as pd

from SSc_Somatic import configure, load_config, load_study
from SSc_Somatic.loading import load_signature_profiles
from SSc_Somatic.signatures import analysis as sig_mut_funcs
from SSc_Somatic.signatures import plots as sig_mut_plots_funcs

# Study configuration. Point this at another configuration file to analyze a different cohort.
config = configure(load_config(Path('..') / 'config' / 'example_SSc.yaml'))
data = load_study(config, 'signatures')

# Cohort labels, clinical columns and signature names used below
group_col = config.require('cohort', 'group_column')
control_label = config.require('cohort', 'control_label')
subtype_a, subtype_b = config.require('cohort', 'subtypes')
clinical = config.require('cohort', 'clinical_columns')
age_col = clinical['age']
duration_col = clinical['disease_duration']
dlco_col = clinical['dlco_percent_predicted']
fvc_col = clinical['fvc_percent_predicted']
mrss_col = clinical['skin_score']
ethnicity_col = config.require('cohort', 'ethnicity_column')
clinical_outcomes = [dlco_col, fvc_col, mrss_col]
denovo_sbs = config.require('signatures', 'denovo_sbs')
cosmic_sbs = config.require('signatures', 'cosmic_sbs')
denovo_indel = config.require('signatures', 'denovo_indel')
baseline_duration = config.require('signatures', 'baseline_duration')

# Annotated sample table, sample ID lists and SigProfiler output
meta_df = data.samples
ssc_ids = data.case_ids
normal_ids = data.control_ids
limited_ids = data.group_ids[subtype_a]
diffuse_ids = data.group_ids[subtype_b]
mat_sbs_df = data.signatures['mat_sbs_df']
mat_id_df = data.signatures['mat_id_df']
ex_sbs_df = data.signatures['ex_sbs_df']
ex_id_df = data.signatures['ex_id_df']
assign_sbs_df = data.signatures['assign_sbs_df']
anno_ex_sbs_df = data.signatures['anno_ex_sbs_df']
anno_assign_sbs_df = data.signatures['anno_assign_sbs_df']

denovo_sig_path = config.path('signatures', 'denovo_signature_profiles')
decompose_cosmic_sig_path = config.path('signatures', 'cosmic_signature_profiles')
plot_dir = str(config.output_dir('signatures', 'figures_dir')) + os.sep

In [ ]:
# Assign race and ethnicity categories and show their counts by disease group, then load
# the de novo and COSMIC signature profiles
meta_df['race_eth'] = meta_df.apply(sig_mut_funcs.assign_race_ethnicity, axis=1)
print("Number of participants by race and ethnicity category and disease group:")
print(pd.crosstab(meta_df['race_eth'], meta_df[group_col], margins=True))

denovo_df = load_signature_profiles(denovo_sig_path)
cosmic_df = load_signature_profiles(decompose_cosmic_sig_path)

In [ ]:
# Compute SBS and indel counts per sample by disease group
variant_data, variant_stats_df = sig_mut_funcs.compute_variant_stats_by_group(
    mat_sbs_df, mat_id_df, normal_ids, limited_ids, diffuse_ids
)
print("SBS and indel counts per sample by disease group:")
print(variant_stats_df.to_string(index=False))

In [ ]:
# Plot SBS and indel counts per sample by disease group
print("Figure: SBS and indel counts per sample by disease group")
fig, ax = sig_mut_plots_funcs.plot_variant_counts_by_group(
    variant_data, variant_stats_df, output_filename=plot_dir + 'fig4a_variant_counts'
)

In [ ]:
# Z-score the SBS and indel counts and annotate them with metadata
mutation_data, zscore_stats_df = sig_mut_funcs.compute_zscore_mutation_data(
    mutation_dfs={'SBS': mat_sbs_df, 'Indel': mat_id_df},
    meta_df=meta_df,
    sample_id_from_column=True,
    sample_id_delimiter='_',
    sample_id_position=0
)
print("Summary statistics of z-scored SBS and indel counts:")
print(zscore_stats_df.to_string(index=False))

In [ ]:
# Compare z-scored mutation counts between case subtypes and age groups, and plot them
print("Figure: z-scored SBS and indel counts by case subtype and age group")
comparison_results = sig_mut_funcs.compute_subtype_comparison_stats(mutation_data)
fig, axes = sig_mut_plots_funcs.plot_zscore_by_subtype_age(
    mutation_data, comparison_results, output_filename=plot_dir + 'fig4b_zscore_boxplots'
)

In [ ]:
# Compare de novo SBS signature activity between disease groups, and plot it
print("Figure: activity of the de novo SBS signatures by disease group")
sig_stats = sig_mut_funcs.compute_signature_stats(anno_ex_sbs_df, denovo_sbs)
fig, axes = sig_mut_plots_funcs.plot_signature_distributions(
    anno_ex_sbs_df, denovo_sbs, sig_stats, output_filename=plot_dir + 'fig4c_denovo_sbs'
)

In [ ]:
# Compare COSMIC-assigned SBS signature activity between disease groups, and plot it
print("Figure: activity of the COSMIC-assigned SBS signatures by disease group")
sig_stats = sig_mut_funcs.compute_signature_stats(anno_assign_sbs_df, cosmic_sbs)
fig, axes = sig_mut_plots_funcs.plot_signature_distributions(
    anno_assign_sbs_df, cosmic_sbs, sig_stats, output_filename=plot_dir + 'fig4d_assigned_sbs'
)

In [ ]:
# Compare de novo signature activity between cases and controls by race, cohort
# (baseline and early disease, young age) and comparison
print("Comparing de novo signature activity between cases and controls:")
comprehensive_results = sig_mut_funcs.run_comprehensive_disease_analysis(
    meta_df=meta_df,
    signature_dfs={'ex_sbs': ex_sbs_df, 'ex_id': ex_id_df},
    baseline_duration=baseline_duration,
    early_duration=config.require('signatures', 'early_duration'),
    young_age=config.require('signatures', 'young_age'),
    exclude_other=True,
    verbose=True
)

In [ ]:
# Prepare volcano plot data from comprehensive analysis results
volcano_df = sig_mut_funcs.prepare_volcano_disease_data(comprehensive_results)

# Plot volcano signatures elevated in disease (fold_change > 1)
print("Figure: volcano plot of signatures elevated in disease, across comparisons, race groups "
      "and cohorts")
fig, ax = sig_mut_plots_funcs.plot_volcano_disease_comparisons(
    volcano_df,
    min_fold_change=1.0,
    fdr_threshold=0.05,
    fc_label_threshold=1.2,
    pval_label_threshold=0.75,
    output_filename=plot_dir + 'fig4_volcano_disease_comparisons'
)

In [ ]:
# Clinical effects by group, for de novo and COSMIC-assigned SBS signatures.
# Each signature set is (activity table, signature names).
signature_sets = {
    'denovo': (ex_sbs_df, denovo_sbs),
    'cosmic_sbs': (assign_sbs_df, cosmic_sbs),
}
effect_metadata_cols = clinical_outcomes + [duration_col, age_col, 'Black', ethnicity_col,
                                            'ana', 'anti-centromere', 'anti-topo']

print(f"Figures (shown at the end of this cell): effect sizes of clinical variables on SBS "
      f"signature activity, for de novo and then COSMIC-assigned signatures, each in the "
      f"{subtype_a}, {subtype_b} and {control_label} groups")
effect_results = {}
for set_name, (signature_df, signatures) in signature_sets.items():
    print(f"Clinical effect analysis by disease group, {'de novo' if set_name == 'denovo' else 'COSMIC-assigned'} SBS signatures:")
    effect_results[set_name] = sig_mut_funcs.run_clinical_effect_analysis_by_subtype(
        signature_df, meta_df,
        signature_cols=signatures,
        metadata_cols=effect_metadata_cols
    )
    for group in (subtype_a, subtype_b, control_label):
        fig, axes = sig_mut_plots_funcs.plot_clinical_effect_bars(
            effect_results[set_name][group]['effect_df'],
            output_filename=plot_dir + f'{set_name}_{group.lower()}_effects'
        )

In [ ]:
# Clinical effect heatmaps for each signature set
print("Figures: heatmaps of clinical variable associations with SBS signature activity by "
      "disease group, for de novo and then COSMIC-assigned signatures")
for set_name, (_, signatures) in signature_sets.items():
    fig, ax = sig_mut_plots_funcs.plot_clinical_effect_heatmap(
        effect_results[set_name],
        signatures=signatures,
        continuous_vars=[age_col, duration_col, mrss_col, dlco_col, fvc_col],
        categorical_vars=['Black', ethnicity_col, 'ana', 'anti-centromere', 'anti-topo'],
        output_filename=plot_dir + f'fig4_clinical_effect_heatmap_{set_name}'
    )

In [ ]:
# Correlations between signature activity and clinical outcomes, summarized and
# plotted for the baseline and full cohorts. De novo correlations include the
# indel signatures.
correlation_sets = {
    'denovo': ({'ex_sbs': ex_sbs_df, 'ex_id': ex_id_df}, denovo_sbs + denovo_indel),
    'cosmic_sbs': ({'assign_sbs': assign_sbs_df}, cosmic_sbs),
}

print("Figures (shown at the end of this cell): signature activity against clinical outcomes "
      "with linear trend lines, in the baseline and then the full cohort, for de novo and then "
      "COSMIC-assigned signatures")
correlation_results = {}
for set_name, (signature_dfs, signatures) in correlation_sets.items():
    print(f"Correlations of {'de novo' if set_name == 'denovo' else 'COSMIC-assigned'} signature activity with clinical outcomes:")
    correlation_results[set_name] = sig_mut_funcs.compute_signature_clinical_correlations(
        meta_df=meta_df,
        signature_dfs=signature_dfs,
        signatures=signatures,
        clinical_vars=clinical_outcomes,
        baseline_duration=baseline_duration,
        verbose=True
    )
    print(f"Summary of {'de novo' if set_name == 'denovo' else 'COSMIC-assigned'} signature correlations (FDR below 0.1, nominal p below 0.05):")
    sig_mut_funcs.summarize_signature_clinical_correlations(
        correlation_results[set_name]['stats_df'],
        fdr_threshold=0.1,
        nominal_threshold=0.05
    )
    for cohort in ('baseline', 'full'):
        fig, axes = sig_mut_plots_funcs.plot_signature_clinical_correlations(
            correlation_results[set_name],
            cohort=cohort,
            show_ci=True,
            output_filename=plot_dir + f'fig4_correlation_{cohort}_cohort_{set_name}'
        )

In [ ]:
# Compare the de novo signatures with COSMIC signatures by cosine similarity
print("Computing cosine similarity between de novo and COSMIC signature profiles:")
cosine_results = sig_mut_funcs.compute_cosine_similarity_matrix(denovo_df, cosmic_df)
print("Figure: cosine similarity between de novo and COSMIC signatures")
fig, ax = sig_mut_plots_funcs.plot_cosine_similarity_heatmap(
    cosine_results, orientation='tall', output_filename=plot_dir + 'fig4b_cosine_similarity'
)

In [ ]:
# Forest plots of COSMIC-assigned SBS signatures against clinical outcomes, for
# each case subtype and all cases, in the full cohort and the baseline cohort
forest_groups = {
    subtype_a.lower(): (subtype_a, limited_ids),
    subtype_b.lower(): (subtype_b, diffuse_ids),
    'all': ('All cases', ssc_ids),
}
forest_runs = [
    (subtype_a.lower(), 'full'), (subtype_b.lower(), 'full'), ('all', 'full'),
    ('all', 'baseline'), (subtype_a.lower(), 'baseline'), (subtype_b.lower(), 'baseline'),
]

print("Figures (shown at the end of this cell): unadjusted correlations and age-adjusted "
      "standardized betas of COSMIC-assigned SBS signatures with clinical outcomes, in this "
      "order:")
print('\n'.join(f'  {forest_groups[group][0]}, {cohort} cohort' for group, cohort in forest_runs))
forest_stats = {}
for group, cohort in forest_runs:
    label, sample_ids = forest_groups[group]
    cohort_options = {'baseline_duration': baseline_duration} if cohort == 'baseline' else {}
    forest_stats[group, cohort] = sig_mut_funcs.compute_signature_clinical_forest_stats(
        meta_df=meta_df,
        signature_dfs={'sbs': assign_sbs_df},
        signatures=cosmic_sbs,
        clinical_vars=clinical_outcomes,
        sample_ids=sample_ids,
        **cohort_options
    )
    file_suffix = '_bl' if cohort == 'baseline' else ''
    fig, axes = sig_mut_plots_funcs.plot_forest_unadjusted_and_adjusted(
        forest_stats[group, cohort],
        signatures=cosmic_sbs,
        output_filename=plot_dir + f'fig4c_forest_{group}{file_suffix}'
    )

In [ ]:
# Show the forest plot statistics for all cases in the full cohort
print("Unadjusted and age-adjusted associations of COSMIC-assigned SBS signatures with "
      "clinical outcomes, all cases, full cohort:")
print(forest_stats['all', 'full'])